[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/course-1-deep-learning/ext-06-forecasting-clipping-attention.ipynb)

# Advanced Discussion 6 — honest forecasting, gradient clipping, attention in twelve lines

Reproduces the chapter 6 advanced discussion: baselines and chronological evaluation on a synthetic hourly series, an MLP-versus-LSTM comparison, the leakage trap, gradient clipping, and scaled dot-product attention.

**Optional advanced-discussion lab** for Hands-On Deep Learning with Python (AIBits Academy). Not graded and not needed for the certificate. Everything here runs on the Colab free tier (CPU is enough) with built-in datasets, so no download can fail.

Offline fallback: the series is generated locally (the graded lab uses the Metro Interstate data). Runs in about a minute on CPU.

In [ ]:
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F, time
np.random.seed(0); torch.manual_seed(0)

## 1. a synthetic hourly 'throughput' series: daily + weekly seasonality, slow trend, noise, a few outages

In [ ]:
n = 24 * 7 * 40
t = np.arange(n)
series = (100 + 0.004 * t + 25 * np.sin(2 * np.pi * t / 24 - 1.0) + 12 * np.sin(2 * np.pi * t / 168) + np.random.normal(0, 6, n))
series[np.random.choice(n, 25, replace=False)] *= 0.3                       # sensor dropouts / outages
IN, OUT = 72, 24                                                             # look back 3 days, forecast 24 h ahead
def windows(s):
    X = np.stack([s[i:i + IN] for i in range(len(s) - IN - OUT + 1)])
    Y = np.stack([s[i + IN:i + IN + OUT] for i in range(len(s) - IN - OUT + 1)])
    return X.astype(np.float32), Y.astype(np.float32)
X, Y = windows(series); N = len(X)
tr, va, te = slice(0, int(0.7 * N)), slice(int(0.7 * N), int(0.85 * N)), slice(int(0.85 * N), N)     # chronological split
mae = lambda p, y: float(np.abs(p - y).mean())
persist = np.repeat(X[te][:, -1:], OUT, axis=1)                                # repeat the last observation
seasonal = X[te][:, -24:]                                                      # same hours yesterday
ma7 = np.mean([X[te][:, -24 * k:(-24 * (k - 1) or None)] for k in (1, 2, 3)], axis=0)   # mean of the same hour over the 3 days we can see
print("test MAE  persistence %.2f | seasonal-naive %.2f | 3-day same-hour mean %.2f" % (mae(persist, Y[te]), mae(seasonal, Y[te]), mae(ma7, Y[te])))

## 2. small neural forecasters (scaled with TRAIN statistics only)

In [ ]:
mu, sd = X[tr].mean(), X[tr].std()
S = lambda a: torch.tensor((a - mu) / sd)
Xtr, Ytr, Xva, Yva, Xte, Yte = S(X[tr]), S(Y[tr]), S(X[va]), S(Y[va]), S(X[te]), S(Y[te])
class LSTMF(nn.Module):
    def __init__(s, h=64): super().__init__(); s.rnn = nn.LSTM(1, h, batch_first=True); s.head = nn.Linear(h, OUT)
    def forward(s, x): o, _ = s.rnn(x.unsqueeze(-1)); return s.head(o[:, -1])
class MLPF(nn.Module):
    def __init__(s, h=128): super().__init__(); s.net = nn.Sequential(nn.Linear(IN, h), nn.ReLU(), nn.Linear(h, OUT))
    def forward(s, x): return s.net(x)
def fit(model, epochs=25, lr=2e-3, clip=None):
    opt = torch.optim.Adam(model.parameters(), lr=lr); best, best_state = 1e9, None
    for ep in range(epochs):
        model.train(); perm = torch.randperm(len(Xtr))
        for i in range(0, len(Xtr), 64):
            b = perm[i:i + 64]; opt.zero_grad(); F.l1_loss(model(Xtr[b]), Ytr[b]).backward()
            if clip: nn.utils.clip_grad_norm_(model.parameters(), clip)
            opt.step()
        model.eval()
        with torch.no_grad(): v = F.l1_loss(model(Xva), Yva).item()
        if v < best: best, best_state = v, {k: x.clone() for k, x in model.state_dict().items()}
    model.load_state_dict(best_state); model.eval()
    with torch.no_grad(): p = model(Xte).numpy() * sd + mu
    return p
torch.manual_seed(0); t0 = time.time(); p_mlp = fit(MLPF()); print("MLP  test MAE %.2f (%.0fs)" % (mae(p_mlp, Y[te]), time.time() - t0))
torch.manual_seed(0); t0 = time.time(); p_lstm = fit(LSTMF()); print("LSTM test MAE %.2f (%.0fs)" % (mae(p_lstm, Y[te]), time.time() - t0))
print("improvement of the LSTM over the best baseline: %.0f%%" % (100 * (1 - mae(p_lstm, Y[te]) / min(mae(persist, Y[te]), mae(seasonal, Y[te]), mae(ma7, Y[te])))))
print("error by horizon (hours ahead) LSTM:", [round(float(np.abs(p_lstm[:, h] - Y[te][:, h]).mean()), 1) for h in (0, 5, 11, 17, 23)])

## 3. the leakage trap: a random split of overlapping windows

In [ ]:
rng = np.random.RandomState(0); idx = rng.permutation(N); cut = int(0.7 * N)
tr2, te2 = idx[:cut], idx[cut:]
Xtr2, Ytr2, Xte2, Yte2 = S(X[tr2]), S(Y[tr2]), S(X[te2]), S(Y[te2])
m = MLPF(); opt = torch.optim.Adam(m.parameters(), lr=2e-3); torch.manual_seed(0)
for ep in range(25):
    perm = torch.randperm(len(Xtr2))
    for i in range(0, len(Xtr2), 64):
        b = perm[i:i + 64]; opt.zero_grad(); F.l1_loss(m(Xtr2[b]), Ytr2[b]).backward(); opt.step()
m.eval()
with torch.no_grad(): p2 = m(Xte2).numpy() * sd + mu
print("MLP MAE with a RANDOM split (leaky): %.2f  vs chronological split: %.2f" % (mae(p2, Y[te2]), mae(p_mlp, Y[te])))

## 4. exploding gradients and clipping

In [ ]:
def deep_rnn_grad(clip):
    torch.manual_seed(0)
    rnn = nn.RNN(1, 32, nonlinearity="relu", batch_first=True); head = nn.Linear(32, 1)
    with torch.no_grad(): rnn.weight_hh_l0.copy_(1.0 * torch.eye(32) + 0.02 * torch.randn(32, 32))
    x = torch.randn(64, 200, 1); y_ = torch.randn(64, 1)
    o, _ = rnn(x); loss = F.mse_loss(head(o[:, -1]), y_); loss.backward()
    ps = list(rnn.parameters()) + list(head.parameters())
    pre = torch.nn.utils.clip_grad_norm_(ps, clip)                 # returns the norm BEFORE clipping
    post = torch.sqrt(sum((p.grad ** 2).sum() for p in ps))
    return pre.item(), post.item()
print("gradient norm on a 200-step ReLU RNN (recurrent weights ~ identity): before clipping %.3g, after clip_grad_norm_(1.0) %.3g" % deep_rnn_grad(1.0))

## 5. attention in ~12 lines

In [ ]:
def attention(Q, K, V, mask=None):
    scores = Q @ K.transpose(-2, -1) / Q.shape[-1] ** 0.5
    if mask is not None: scores = scores.masked_fill(~mask, float("-inf"))
    w = torch.softmax(scores, dim=-1)
    return w @ V, w
torch.manual_seed(0)
q = torch.randn(1, 3, 4); k = torch.randn(1, 5, 4); v = torch.randn(1, 5, 4)
out, w = attention(q, k, v)
print("attention weights rows sum to", w.sum(-1).round(decimals=3).tolist(), "| output shape", tuple(out.shape))
causal = torch.tril(torch.ones(4, 4, dtype=torch.bool))
_, wc = attention(torch.randn(1, 4, 4), torch.randn(1, 4, 4), torch.randn(1, 4, 4), causal)
print("causal weights (row i may only look at columns <= i):"); print(wc[0].round(decimals=2))

## Your turn

Change one setting at a time (seed, learning rate, ε, perplexity, threshold), re-run, and write one sentence about what changed and why. Keep the numbers: your results will differ slightly from the chapter's, and that difference is worth explaining.